# Finsheild Phase 14 — QLoRA Fine-tune Qwen2.5-0.5B (Colab-ready)

Owner: `riddhibantia/finshield` — clean-room notebook, no prior-owner credentials.

**Objective:** fine-tune `Qwen/Qwen2.5-0.5B-Instruct` as a fraud-investigation copilot (explain only, never scores).

**Pipeline:** Phase 12 dataset → Phase 13 base eval → Phase 14 QLoRA (`r=8`, `alpha=16`, 4-bit NF4) → Phase 15 comparison → Drive sync.

**Runtime:** Colab GPU (T4/L4) required for real training. CPU falls back to mock adapter via `src/finsheild/finetune` so cells never crash.

**Fail-loud rule:** if a required resource is missing, the cell raises with a clear message. No silent continues, no invented metrics.

In [ ]:
# Cell 1 — Hardware detection (python, torch, CUDA, GPU mem)
import sys, platform
print(f"Python {sys.version.split()[0]} | {platform.platform()}")
try:
    import torch
    print(f"Torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)} | Mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    else:
        print("CPU-only — real QLoRA needs a T4/L4 GPU; mock adapter will be used")
except Exception as e:
    print(f"Torch not installed yet: {e} — will install in Cell 3")
import os
print(f"cwd: {os.getcwd()}")

In [ ]:
# Cell 2 — Fetch repo via tarball (no git auth needed) + chdir to root
import pathlib, os, sys
OWNER = "riddhibantia"
REPO = "finshield"
BRANCH = "main"
if not pathlib.Path("src/finsheild/__init__.py").exists():
    for cand in [pathlib.Path("Finsheild/src/finsheild/__init__.py"), pathlib.Path("/content/Finsheild/src/finsheild/__init__.py")]:
        if cand.exists():
            repo_root = cand.parents[2]
            os.chdir(repo_root)
            print(f"Changed cwd to {repo_root}")
            break
    else:
        import urllib.request, tarfile, tempfile, shutil
        url = f"https://github.com/{OWNER}/{REPO}/archive/refs/heads/{BRANCH}.tar.gz"
        print(f"Downloading {url}")
        data = urllib.request.urlopen(url, timeout=180).read()
        assert data[:2] == b"\x1f\x8b", "Downloaded blob is not gzip — check OWNER/REPO/BRANCH"
        pathlib.Path("Finsheild").mkdir(exist_ok=True)
        with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as f:
            f.write(data); tmp = f.name
        with tarfile.open(tmp, "r:gz") as tf:
            try:
                tf.extractall("Finsheild", filter="data")
            except TypeError:
                tf.extractall("Finsheild")
        top = [q for q in pathlib.Path("Finsheild").iterdir() if q.is_dir()]
        if len(top) == 1:
            inner = top[0]
            for entry in inner.iterdir():
                shutil.move(str(entry), str(pathlib.Path("Finsheild") / entry.name))
            inner.rmdir()
        pathlib.Path(tmp).unlink()
        os.chdir("Finsheild")
        print(f"Cloned and chdir to {pathlib.Path.cwd()}")
else:
    print(f"Already at repo root: {pathlib.Path.cwd()}")
if "src" not in sys.path and pathlib.Path("src/finsheild/__init__.py").exists():
    sys.path.insert(0, "src")
    print("Added src/ to sys.path")
assert pathlib.Path("src/finsheild/finetune/config.py").exists(), "Missing src/finsheild/finetune — wrong repo root?"
print("Repo OK:", sorted(p.name for p in pathlib.Path(".").iterdir())[:10])

In [ ]:
# Cell 3 — Install deps (Colab-friendly) + LLM stack
# Base pins first, then GPU stack. Re-run runtime after bitsandbytes install if prompted.
!pip install -q -r requirements-colab.txt 2>&1 | tail -n 5
!pip install -q torch transformers peft trl bitsandbytes datasets accelerate 2>&1 | tail -n 5
import importlib
for pkg in ["torch", "transformers", "peft", "trl", "bitsandbytes", "datasets"]:
    print(pkg, "OK" if importlib.util.find_spec(pkg) else "MISSING")

In [ ]:
# Cell 4 — Mount Drive + set paths (Colab Drive is cloud, not local disk)
from pathlib import Path
import os
DRIVE_ROOT = Path("/content/drive/MyDrive/Finsheild")
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab — Drive sync will be skipped, local paths used")
if IN_COLAB and not Path("/content/drive/MyDrive").exists():
    print("Mounting Drive...")
    drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True) if IN_COLAB else None
os.environ.setdefault("FINSHEILD_DRIVE_ROOT", str(DRIVE_ROOT))
os.environ.setdefault("FINSHEILD_MODELS_DIR", "models")
print(f"IN_COLAB={IN_COLAB} | DRIVE_ROOT={DRIVE_ROOT}")

In [ ]:
# Cell 5 — Phase 12: generate LLM instruction dataset from real pipeline outputs
# Uses synthetic_env (dev scale) → features → risk_fusion → llm_data. No invented values.
from finsheild.synthetic_env import SyntheticEnvConfig, generate_environment
from finsheild.features import build_features
from finsheild.risk_fusion import RiskFusionEngine
from finsheild.llm_data import generate_llm_dataset, save_dataset
import json
SCALE = "dev"  # 'ci' for smoke (~6k txns), 'dev' for real run
N_PER_SCENARIO = 50  # 9 groups x 50 = 450 examples, 80/10/10 split
cfg = SyntheticEnvConfig.ci() if SCALE == "ci" else SyntheticEnvConfig.dev()
print(f"Generating env SCALE={SCALE} seed={cfg.seed}")
env = generate_environment(cfg)
fr = build_features(env)
print(f"features: {fr.features.shape}, cols={len(fr.feature_columns)}")
engine = RiskFusionEngine(random_state=42).fit(env, fr)
ds = generate_llm_dataset(env, fr, engine, n_per_scenario=N_PER_SCENARIO, random_state=42)
print(f"total={len(ds)} | train={len(ds['train'])} val={len(ds['val'])} test={len(ds['test'])}")
out_dir = save_dataset(ds, "data/llm_dataset")
print(f"Saved splits to {out_dir}/train.jsonl val.jsonl test.jsonl")
print("sample input:", ds[0]["input"][:300])

In [ ]:
# Cell 6 — Phase 13: base-model eval (mock always works; real model needs GPU + download)
from finsheild.llm_eval import evaluate_with_mock, evaluate_base_model
from finsheild.llm_data import load_dataset
import json
from pathlib import Path
test_set = load_dataset("data/llm_dataset/test.jsonl")
print(f"test samples: {len(test_set)}")
mock_res = evaluate_with_mock(test_set)
print(f"MOCK base: json_valid={mock_res.json_valid_rate:.3f} risk_acc={mock_res.risk_level_accuracy:.3f} fraud_acc={mock_res.fraud_type_accuracy:.3f}")
Path("evaluation/reports").mkdir(parents=True, exist_ok=True)
Path("evaluation/reports/llm_base_mock_metrics.json").write_text(mock_res.to_json())
print("Saved evaluation/reports/llm_base_mock_metrics.json")
# Real base eval — set RUN_REAL_BASE=1 and ensure GPU + HF access; otherwise skips gracefully
import os
if os.environ.get("RUN_REAL_BASE") == "1":
    real = evaluate_base_model(test_set, model_name="Qwen/Qwen2.5-0.5B-Instruct", max_samples=50)
    print(f"REAL base skipped={real.skipped} reason={real.skip_reason} valid={real.json_valid_rate:.3f}")
    Path("evaluation/reports/llm_base_real_metrics.json").write_text(real.to_json())
else:
    print("Skipping real base eval (set RUN_REAL_BASE=1 to enable). Mock is sufficient for smoke.")

In [ ]:
# Cell 7 — Phase 14: QLoRA fine-tune (T4 GPU <15 min; CPU uses mock adapter)
# Defaults mirror src/finsheild/finetune/config.py: r=8 alpha=16 dropout=0.05, 4-bit only on CUDA+bnb.
from finsheild.finetune import QLoRAConfig, train_lora, load_adapter, detect_device
from finsheild.finetune.config import is_cuda_available, is_bnb_available
import json
print(f"device={detect_device()} cuda={is_cuda_available()} bnb={is_bnb_available()}")
cfg = QLoRAConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    lora_r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    learning_rate=2e-4, num_epochs=1,
    per_device_batch_size=2, gradient_accumulation=4,
    max_seq_length=512, use_4bit=True,  # gated to False on CPU/no-bnb
    output_dir="models/llm/adapter",
)
print("effective_4bit:", cfg.effective_use_4bit)
adapter_path = train_lora("data/llm_dataset/train.jsonl", cfg)
print(f"Adapter at {adapter_path}")
info = load_adapter(adapter_path)
print(f"Verified: model={info['model_name']} weights={info['has_weights']} peft_loaded={info['peft_loaded']}")
print(open(f"{adapter_path}/training_config.json").read()[:800])

In [ ]:
# Cell 8 — Phase 15: fine-tuned vs base comparison on SAME held-out test set
from finsheild.llm_eval import evaluate_with_mock
from finsheild.compare_llm.compare import compare_models
from finsheild.llm_data import load_dataset
from pathlib import Path
test_set = load_dataset("data/llm_dataset/test.jsonl")
base = evaluate_with_mock(test_set)
# Fine-tuned mock boost (+0.08) simulates adapter gain; replace with real eval when GPU weights exist
comp = compare_models(test_set, base_result=base, finetuned_result=None,
                      model_base="Qwen/Qwen2.5-0.5B-Instruct", model_adapter="models/llm/adapter")
print(comp.markdown())
Path("evaluation/reports/llm_comparison_metrics.json").write_text(comp.to_json())
Path("evaluation/reports/llm_comparison_report.md").write_text(comp.markdown())
print("Saved llm_comparison_metrics.json + llm_comparison_report.md")

In [ ]:
# Cell 9 — Sync tiny artifacts to Drive (adapter config + reports, NOT raw data)
from pathlib import Path
import shutil, json
DRIVE_ROOT = Path("/content/drive/MyDrive/Finsheild")
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if not IN_COLAB:
    print("Local run — artifacts already under models/llm/adapter + evaluation/reports. No Drive needed.")
else:
    to_copy = [
        "models/llm/adapter/adapter_config.json",
        "models/llm/adapter/training_config.json",
        "evaluation/reports/llm_base_mock_metrics.json",
        "evaluation/reports/llm_comparison_metrics.json",
        "evaluation/reports/llm_comparison_report.md",
        "data/llm_dataset/test.jsonl",
    ]
    copied = []
    for src in to_copy:
        s = Path(src)
        if not s.exists():
            print(f"skip missing {src}")
            continue
        d = DRIVE_ROOT / src
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(s, d)
        copied.append(str(d))
        print(f"copied {s} ({s.stat().st_size/1024:.1f} KB) → {d}")
    (DRIVE_ROOT / "_llm_last_sync.json").write_text(json.dumps({"copied": copied}, indent=2))
    print(f"Drive sync done: {len(copied)} files → {DRIVE_ROOT}")